

---


# 🛒 E-commerce Data Processing & Analysis Using Pandas

## 📌 Project Overview

This project demonstrates how **Python and Pandas** can be used to combine, clean, transform, and prepare e-commerce data for analysis.

Three related datasets are used:

- **Orders** — order-level transaction information
- **Customers** — customer details
- **Products** — product information

### What this notebook demonstrates

- Loading CSV files with `read_csv()`
- Inspecting datasets and their structure
- Combining DataFrames with `concat()`
- Joining related datasets using `merge()`
- Creating calculated fields with `apply()`
- Performing DateTime transformations
- Extracting useful date attributes
- Checking data quality
- Organizing a clean analytical dataset
- Exporting the processed data to CSV

> **Goal:** Transform raw e-commerce tables into a clean, structured dataset ready for analysis and reporting.


---



## 1. Import Libraries

In [16]:
import pandas as pd
from pathlib import Path

## 2. Load the Three CSV Files

The files should be placed in the same folder as this notebook.

In [17]:
orders = pd.read_csv("Orders.csv")
customers = pd.read_csv("Customers.csv")
products = pd.read_csv("Products.csv")

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Products shape:", products.shape)

Orders shape: (120, 7)
Customers shape: (30, 5)
Products shape: (20, 5)


## 3. Inspect the Datasets

In [18]:
display(orders.head())
display(customers.head())
display(products.head())

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


In [19]:
print("Orders columns:", orders.columns.tolist())
print("Customers columns:", customers.columns.tolist())
print("Products columns:", products.columns.tolist())

Orders columns: ['Order_ID', 'Order_Date', 'Customer_ID', 'Product_ID', 'Quantity', 'Payment_Method', 'Order_Status']
Customers columns: ['Customer_ID', 'Customer_Name', 'City', 'Region', 'Membership_Type']
Products columns: ['Product_ID', 'Product_Name', 'Category', 'Unit_Price', 'Brand']


## 4. Demonstrate `concat()`

`concat()` is demonstrated by vertically combining two DataFrames with a common structure.

In [20]:
customer_demo = customers[["Customer_ID", "Customer_Name"]].copy()
customer_demo["Source"] = "Customers"

product_demo = products[["Product_ID", "Product_Name"]].copy()
product_demo = product_demo.rename(columns={
    "Product_ID": "Customer_ID",
    "Product_Name": "Customer_Name"
})
product_demo["Source"] = "Products"

concat_demo = pd.concat(
    [customer_demo, product_demo],
    ignore_index=True
)

display(concat_demo.head(10))
print("Combined rows using concat():", len(concat_demo))

,Customer_ID,Customer_Name,Source
0,C001,Aarav Sharma,Customers
1,C002,Zoya Khan,Customers
2,C003,Rohan Mehta,Customers
3,C004,Ananya Singh,Customers
4,C005,Kabir Ali,Customers
5,C006,Ishita Gupta,Customers
6,C007,Aditya Verma,Customers
7,C008,Sara Ahmed,Customers
8,C009,Arjun Nair,Customers
9,C010,Mehak Bhat,Customers


Combined rows using concat(): 50


## 5. Merge Orders with Customers and Products

The common keys are `Customer_ID` and `Product_ID`.

In [21]:
processed = orders.merge(
    customers,
    on="Customer_ID",
    how="left"
)

processed = processed.merge(
    products,
    on="Product_ID",
    how="left"
)

print("Merged dataset shape:", processed.shape)
display(processed.head())

Merged dataset shape: (120, 15)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


## 6. DateTime Operations

Convert `Order_Date` to a DateTime type and extract year, month, day, week, and day name.

In [22]:
processed["Order_Date"] = pd.to_datetime(
    processed["Order_Date"],
    errors="coerce"
)

processed["Order_Year"] = processed["Order_Date"].dt.year
processed["Order_Month"] = processed["Order_Date"].dt.month
processed["Order_Month_Name"] = processed["Order_Date"].dt.strftime("%B")
processed["Order_Day"] = processed["Order_Date"].dt.day
processed["Order_Day_Name"] = processed["Order_Date"].dt.day_name()
processed["Order_Week"] = processed["Order_Date"].dt.isocalendar().week.astype(int)

display(processed[[
    "Order_Date", "Order_Year", "Order_Month",
    "Order_Month_Name", "Order_Day", "Order_Day_Name", "Order_Week"
]].head())

,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Order_Week
0,2026-02-19,2026,2,February,19,Thursday,8
1,2026-01-25,2026,1,January,25,Sunday,4
2,2026-02-26,2026,2,February,26,Thursday,9
3,2026-03-04,2026,3,March,4,Wednesday,10
4,2026-03-29,2026,3,March,29,Sunday,13


## 7. Use `apply()` to Create Total Amount

`Total_Amount` is calculated as Quantity × Unit Price.

In [23]:
processed["Total_Amount"] = processed.apply(
    lambda row: row["Quantity"] * row["Unit_Price"],
    axis=1
)

display(processed[[
    "Order_ID", "Product_Name", "Quantity", "Unit_Price", "Total_Amount"
]].head(10))

,Order_ID,Product_Name,Quantity,Unit_Price,Total_Amount
0,O0001,Cricket Bat,2,2499,4998
1,O0002,Wireless Mouse,2,899,1798
2,O0003,Smart Watch,1,3299,3299
3,O0004,Machine Learning Basics,3,999,2997
4,O0005,Coffee Maker,5,3499,17495
5,O0006,Football,3,799,2397
6,O0007,Wireless Mouse,3,899,2697
7,O0008,Power Bank,4,1199,4796
8,O0009,Coffee Maker,2,3499,6998
9,O0010,Power Bank,2,1199,2398


## 8. Organize the Final DataFrame

In [24]:
final_columns = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month",
    "Order_Month_Name", "Order_Day", "Order_Day_Name", "Order_Week",
    "Customer_ID", "Customer_Name", "City", "Region", "Membership_Type",
    "Product_ID", "Product_Name", "Category", "Brand", "Unit_Price",
    "Quantity", "Total_Amount", "Payment_Method", "Order_Status"
]

processed = processed[final_columns]
display(processed.head())
print("Final shape:", processed.shape)

,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Order_Week,Customer_ID,Customer_Name,...,Membership_Type,Product_ID,Product_Name,Category,Brand,Unit_Price,Quantity,Total_Amount,Payment_Method,Order_Status
0,O0001,2026-02-19,2026,2,February,19,Thursday,8,C027,Harsh Vardhan,...,Premium,P019,Cricket Bat,Sports,BatPro,2499,2,4998,Credit Card,Delivered
1,O0002,2026-01-25,2026,1,January,25,Sunday,4,C006,Ishita Gupta,...,Premium,P003,Wireless Mouse,Electronics,TechGear,899,2,1798,Debit Card,Delivered
2,O0003,2026-02-26,2026,2,February,26,Thursday,9,C015,Karan Joshi,...,Regular,P004,Smart Watch,Electronics,FitTech,3299,1,3299,Cash on Delivery,Delivered
3,O0004,2026-03-04,2026,3,March,4,Wednesday,10,C024,Maryam Khan,...,Regular,P015,Machine Learning Basics,Books,AIPress,999,3,2997,Net Banking,Delivered
4,O0005,2026-03-29,2026,3,March,29,Sunday,13,C025,Reyansh Jain,...,New,P009,Coffee Maker,Home & Kitchen,HomeBrew,3499,5,17495,Credit Card,Delivered


Final shape: (120, 22)


## 9. Data Quality Check

In [25]:
print("Missing values in each column:")
display(processed.isnull().sum().to_frame("Missing_Values"))

print("Duplicate rows:", processed.duplicated().sum())
print("Total records:", len(processed))

Missing values in each column:


,Missing_Values
Order_ID,0
Order_Date,0
Order_Year,0
Order_Month,0
Order_Month_Name,0
Order_Day,0
Order_Day_Name,0
Order_Week,0
Customer_ID,0
Customer_Name,0


Duplicate rows: 0
Total records: 120


## 10. Export the Final Processed Dataset

The processed DataFrame is saved as `Processed_Ecommerce_Dataset.csv`.

In [26]:
output_file = Path("Processed_Ecommerce_Dataset.csv")
processed.to_csv(output_file, index=False)

print(f"Processed dataset exported successfully: {output_file.resolve()}")

Processed dataset exported successfully: /content/Processed_Ecommerce_Dataset.csv




---


## 🔎 Key Observations

Based on the processed dataset and the transformations performed in this notebook:

1. **Data integration was successful:** Orders, Customers, and Products were combined using their related identifiers, creating a single analytical view.
2. **The final dataset contains 120 records and 23 columns**, including the additional `Order_Value_Band` field.
3. **Customer and product attributes are available alongside each order**, allowing analysis by customer, location, membership type, product, category, and brand.
4. **DateTime processing adds useful time dimensions**, including year, month, month name, day, day name, and week number.
5. **`Total_Amount` provides a useful sales metric**, calculated from `Quantity × Unit_Price`.
6. **`Order_Value_Band` groups transactions into Low, Medium, and High value categories**, supporting basic transaction segmentation.
7. **The final structure is analysis-ready**, with organized columns and standardized values.

> These observations describe the dataset structure and processing results. Specific business trends should only be reported after calculating the corresponding metrics.


---





---


## ✅ Conclusion

This project successfully transformed three separate e-commerce datasets into one **clean, structured, and analysis-ready dataset** using Pandas.

The workflow demonstrated practical data-processing techniques including **`concat()`, `merge()`, `apply()`, DateTime operations, data-quality checks, column organization, and CSV export**.

The resulting dataset combines order, customer, and product information in a format suitable for further **exploratory data analysis, visualization, sales reporting, customer segmentation, and business intelligence**.

### 🚀 Possible Next Steps

- Analyze sales by category and product
- Identify top-performing products and brands
- Compare sales across cities and regions
- Analyze monthly and weekly sales patterns
- Compare customer membership types
- Create visualizations using Matplotlib or Seaborn
- Build a dashboard using Power BI or Tableau


---


